### FUNÇÕES E BLIBIOTECAS

In [1]:
import pandas as pd
from extractor import extract_all_files, inspect
from storage import save_csv, save_parquet, load_parquet

### CAMINHOS E CONFIGS GERAIS

In [2]:
RAW_PATH       = r"C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Raw"
PROCESSED_PATH = r"C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed"

In [3]:
files = {
        "deliveries": {
            "file": "deliveries.csv",
            "params": {"sep": ",", "encoding": "utf-8"},
            "schema": [
                "delivery_id", "driver_id", "hub_id", "status",
                "shipped_date", "delivered_date", "distance_km",
                "cost", "customer_rating", "created_at", "updated_at"
            ]
        },
        "drivers": {
            "file": "drivers.csv",
            "params": {"sep": ",", "encoding": "utf-8"},
            "schema": ["driver_id", "name", "hub_id"]
        }
    }


### ELT

EXTRAÇÃO

In [4]:
dfs = extract_all_files(RAW_PATH, files)

[extract] deliveries.csv | 102000 linhas | 11 colunas
[extract] drivers.csv | 255 linhas | 3 colunas


In [5]:
if "deliveries" in dfs:
    inspect(dfs["deliveries"], "deliveries")

if "drivers" in dfs:
    inspect(dfs["drivers"], "drivers")


[inspect] deliveries

Shape: (102000, 11)
Memória: 14.11 MB

Colunas:
['delivery_id', 'driver_id', 'hub_id', 'status', 'shipped_date', 'delivered_date', 'distance_km', 'cost', 'customer_rating', 'created_at', 'updated_at']

Resumo estrutural:
                   dtype  nulls  n_unique
delivery_id        int64      0    100000
driver_id          int64      0       302
hub_id             int64      0        10
status               str      0         3
shipped_date         str      0       121
delivered_date       str  37283       144
distance_km          str      0     62379
cost             float64      0     64078
customer_rating  float64   5017        42
created_at           str      0       126
updated_at           str      0       126

Estatísticas numéricas:
          delivery_id      driver_id         hub_id   status shipped_date  \
count   102000.000000  102000.000000  102000.000000   102000       102000   
unique            NaN            NaN            NaN        3          121

LOAD

In [6]:
print("\n=== ELT: salvar bruto primeiro ===")
for name, df in dfs.items():
    save_parquet(df, f"{PROCESSED_PATH}\\elt_{name}_raw.parquet")



=== ELT: salvar bruto primeiro ===


[storage] Parquet salvo em 'C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed\elt_deliveries_raw.parquet' — 102000 linhas
[storage] Parquet salvo em 'C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed\elt_drivers_raw.parquet' — 255 linhas


TRANSFORM, puxa ele denovo pra usar

In [7]:
df_elt = load_parquet(f"{PROCESSED_PATH}\\elt_deliveries_raw.parquet")

print(f"[elt] Antes da transformação: {len(df_elt)} linhas")
df_elt = df_elt[df_elt["status"] != "canceled"]

df_elt = df_elt.dropna(subset=["delivered_date"])
print(f"[elt] Após a transformação: {len(df_elt)} linhas")

[storage] Parquet carregado de 'C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed\elt_deliveries_raw.parquet' — 102000 linhas
[elt] Antes da transformação: 102000 linhas
[elt] Após a transformação: 64717 linhas


### ETL

EXTRAÇÃO, basicamente copiei o arquivo ja puxado e salvei em outro dataset

In [8]:
df_etl = dfs["deliveries"].copy()

TRANSFORM

In [9]:
print(f"[etl] Antes da transformação: {len(df_etl)} linhas")
df_etl = df_etl[df_etl["status"] != "canceled"]
df_etl = df_etl.dropna(subset=["delivered_date"])

print(f"[etl] Depois da transformação: {len(df_etl)} linhas")

[etl] Antes da transformação: 102000 linhas
[etl] Depois da transformação: 64717 linhas


In [10]:
df_etl["shipped_date"]   = pd.to_datetime(df_etl["shipped_date"],   errors="coerce")
df_etl["delivered_date"] = pd.to_datetime(df_etl["delivered_date"], errors="coerce")

In [11]:
df_etl["delivery_days"] = (
    df_etl["delivered_date"] - df_etl["shipped_date"]
).dt.days

In [12]:
df_etl = df_etl[[
    "delivery_id",
    "driver_id",
    "hub_id",
    "status",
    "shipped_date",
    "delivered_date",
    "delivery_days",
    "distance_km",
    "cost",
    "customer_rating"
]]

In [13]:
save_parquet(df_etl, f"{PROCESSED_PATH}\\etl_deliveries_clean.parquet")
save_csv(df_etl,     f"{PROCESSED_PATH}\\etl_deliveries_clean.csv")
print(f"[etl] Shape final: {df_etl.shape}")

[storage] Parquet salvo em 'C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed\etl_deliveries_clean.parquet' — 64717 linhas
[storage] CSV salvo em 'C:/Users/Ceifas/OneDrive\Desktop\data-engineering-portfolio/projects/month02_Pipeline/Data/Processed\etl_deliveries_clean.csv' — 64717 linhas
[etl] Shape final: (64717, 10)


### COMPARAR OS DOIS FLUXOS

In [14]:
print("\n===COMPARAÇÃO ETL vs ELT===")
print(f"{'Fluxo':<8} {'Linhas':<12} {'Colunas':<10} {'Arquivo'}")
print(f"{'ELT':<8} {len(df_elt):<12} {df_elt.shape[1]:<10} elt_deliveries_raw.parquet")
print(f"{'ETL':<8} {len(df_etl):<12} {df_etl.shape[1]:<10} etl_deliveries_clean.parquet")
# O :<X usada nis prints serve pra mudar o espaçamento entre as colunas no print, myo uyil


===COMPARAÇÃO ETL vs ELT===
Fluxo    Linhas       Colunas    Arquivo
ELT      64717        11         elt_deliveries_raw.parquet
ETL      64717        10         etl_deliveries_clean.parquet
